In [11]:
# ────────────────────────────────────────────────────────────────
# Cell: Load one CSV → clean → split → save 3 CSV files (train/val/test)
# ────────────────────────────────────────────────────────────────

import pandas as pd
import re
from pathlib import Path
from sklearn.model_selection import train_test_split

In [12]:
# ────────────────────────────────────────────────────────────────
# 1. Path to your single input CSV
# ────────────────────────────────────────────────────────────────

input_csv = "../Datasets/meraged/final_combined_corpus.csv"           # ← CHANGE THIS to your actual file name

In [13]:
# ────────────────────────────────────────────────────────────────
# 2. Light cleaning function (same as before)
# ────────────────────────────────────────────────────────────────

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return " "
    
    if not text.strip():
        return " "
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    
    text = text.strip()
    return text if text else " "

In [14]:
# ────────────────────────────────────────────────────────────────
# 3. Load → clean → remove duplicates
# ────────────────────────────────────────────────────────────────

print(f"Loading and cleaning: {input_csv}\n")

df = pd.read_csv(input_csv)

# Standardize column names if needed
if 'text' not in df.columns:
    text_col = next((c for c in df.columns if 'text' in c.lower() or 'content' in c.lower()), None)
    if text_col:
        df = df.rename(columns={text_col: 'text'})

if 'label' not in df.columns:
    label_col = next((c for c in df.columns if 'label' in c.lower() or 'target' in c.lower() or 'fake' in c.lower()), None)
    if label_col:
        df = df.rename(columns={label_col: 'label'})

# Clean text
df['text_clean'] = df['text'].apply(clean_text)

# Remove very short rows
before_short = len(df)
df = df[df['text_clean'].str.strip().str.len() >= 20].copy()
print(f"Removed {before_short - len(df)} very short/empty rows")

# ─── Remove exact duplicates based on cleaned text ───
before_dup = len(df)
df = df.drop_duplicates(subset=['text_clean'], keep='first').reset_index(drop=True)
print(f"Removed {before_dup - len(df)} duplicate rows (based on cleaned text)")

print(f"After cleaning & deduplication: {len(df):,} rows")

# Final working columns
df_final = df[['text_clean', 'label']].rename(columns={'text_clean': 'text'})

print("\nLabel distribution after cleaning & dedup:")
print(df_final['label'].value_counts(normalize=True).round(3))

Loading and cleaning: ../Datasets/meraged/final_combined_corpus.csv

Removed 130 very short/empty rows
Removed 2322 duplicate rows (based on cleaned text)
After cleaning & deduplication: 2,928,132 rows

Label distribution after cleaning & dedup:
label
real    0.651
fake    0.349
Name: proportion, dtype: float64


In [15]:
# ────────────────────────────────────────────────────────────────
# 4. Stratified split: train / val / test = 60% / 20% / 20%
# ────────────────────────────────────────────────────────────────

print("\nSplitting data (stratified by label)...")

train_df, temp_df = train_test_split(
    df_final,
    test_size=0.4,
    random_state=42,
    stratify=df_final['label'] if df_final['label'].nunique() > 1 else None
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label'] if temp_df['label'].nunique() > 1 else None
)

print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")


Splitting data (stratified by label)...
Train: 1,756,879 rows
Val:   585,626 rows
Test:  585,627 rows


In [16]:
# ────────────────────────────────────────────────────────────────
# 5. Save exactly 3 CSV files
# ────────────────────────────────────────────────────────────────

save_dir = Path("../Datasets/non_aug")
save_dir.mkdir(exist_ok=True)

# Train
train_path = save_dir / "train.csv"
train_df.to_csv(train_path, index=False, encoding="utf-8")
print(f"Saved: {train_path}  ({len(train_df):,} rows)")

# Validation
val_path = save_dir / "val.csv"
val_df.to_csv(val_path, index=False, encoding="utf-8")
print(f"Saved: {val_path}   ({len(val_df):,} rows)")

# Test
test_path = save_dir / "test.csv"
test_df.to_csv(test_path, index=False, encoding="utf-8")
print(f"Saved: {test_path}  ({len(test_df):,} rows)")

print(f"\nAll files saved in folder:")
print(save_dir.resolve())
print("\nEach file contains columns:  text, label")

Saved: ..\Datasets\non_aug\train.csv  (1,756,879 rows)
Saved: ..\Datasets\non_aug\val.csv   (585,626 rows)
Saved: ..\Datasets\non_aug\test.csv  (585,627 rows)

All files saved in folder:
H:\AIML_project\Model_desgin\Datasets\non_aug

Each file contains columns:  text, label
